In [1]:
# ============================================================
# Notebook : 03_causal_stability.ipynb
# Purpose  : Step 3 (Causal Stability) - part 2. Assess how far each
#            category's lag-1 sensor precursor depends on the particular
#            households, WITHOUT claiming independent external replication:
#              (a) A/B split : an INTERNAL stability check (the eligible
#                  households are partitioned into disjoint halves; this is
#                  resampling of one sample, not an independent cohort). A
#                  per-category reproduction verdict is produced for Step 5:
#                  "partial" = precursor sign agrees across halves but the
#                  top precursor sensor differs.
#              (b) Leave-one-household-out (LOGO) resampling -> sign- and
#                  presence-stability of each category's top precursor.
#              (c) Minimum-session-length sensitivity (144 vs 72 steps).
# Design   : Single code cell. No absolute path is printed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.unicode_minus": False,
    "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85",
})
GRAY = ["0.15", "0.40", "0.60", "0.75", "0.88"]


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"

df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
S = meta["sensor_cont"]
ABN = meta["abnormal_categories"]
CAT_LABEL = {"HR": "Heart rate", "BR": "Breath rate", "ST": "Skin temp.", "SP": "SpO2", "ER": "Emergency"}


def partial_corrs(y, X):
    n = len(y)
    Xd = np.column_stack([np.ones(n), X])
    beta, _, _, _ = np.linalg.lstsq(Xd, y, rcond=None)
    resid = y - Xd @ beta
    dof = n - Xd.shape[1]
    s2 = (resid @ resid) / dof
    inv = np.linalg.pinv(Xd.T @ Xd)
    se = np.sqrt(np.maximum(np.diag(inv) * s2, 1e-18))
    tv = beta / se
    return (tv / np.sqrt(tv ** 2 + dof))[1:]


def hh_precursors(min_len):
    """{hh: {cat: {sensor: parcorr}}} over sessions with >= min_len steps."""
    out = {}
    df["_sess_len_"] = df.groupby("session_id")["Timestamp"].transform("size")
    elig = df[df["_sess_len_"] >= min_len]
    for hh, g in elig.groupby("HH_ID"):
        g = g.sort_values("Timestamp")
        if len(g) < min_len:
            continue
        X = g[S].astype(float).values
        keep = X.std(axis=0) > 1e-8
        names = [v for v, k in zip(S, keep) if k]
        Xl = X[:-1][:, keep]
        Xl = (Xl - Xl.mean(0)) / Xl.std(0)
        d = {}
        for c in ABN:
            yv = g["cat_" + c].astype(float).values
            y, ylag = yv[1:], yv[:-1]
            if y.std() < 1e-8 or ylag.std() < 1e-8:
                continue
            pc = partial_corrs(y, np.column_stack([Xl, (ylag - ylag.mean()) / ylag.std()]))[:-1]
            d[c] = {nm: float(v) for nm, v in zip(names, pc)}
        out[hh] = d
    return out


HH = hh_precursors(144)
used_hh = list(HH.keys())
n_used = len(used_hh)


def agg(hh_set):
    acc = {c: {} for c in ABN}
    for hh in hh_set:
        for c, d in HH[hh].items():
            for s, v in d.items():
                acc[c].setdefault(s, []).append(v)
    return {c: {s: float(np.mean(v)) for s, v in d.items()} for c, d in acc.items()}


def top_sensor(d):
    return max(d, key=lambda k: abs(d[k])) if d else None


full = agg(used_hh)

# ---- (a) A/B split & per-category verdict ----
A, B = used_hh[0::2], used_hh[1::2]
pA, pB = agg(A), agg(B)
verdict_rows = []
for c in ABN:
    if c == "ER":
        verdict_rows.append({"category": c, "reproduction": "direct_signal",
                             "top_A": "Button/Shout", "top_B": "Button/Shout", "sign_agreement": "n/a"})
        continue
    da, dbb = pA[c], pB[c]
    ta, tb = top_sensor(da), top_sensor(dbb)
    if not da or not dbb:
        verdict_rows.append({"category": c, "reproduction": "not_reproduced",
                             "top_A": ta or "(none)", "top_B": tb or "(none)", "sign_agreement": "n/a"})
        continue
    common = set(da) & set(dbb)
    sa = float(np.mean([np.sign(da[k]) == np.sign(dbb[k]) for k in common])) if common else 0.0
    if ta == tb and np.sign(da[ta]) == np.sign(dbb[tb]):
        verdict = "full"
    elif sa == 1.0:
        verdict = "partial"
    else:
        verdict = "not_reproduced"
    verdict_rows.append({"category": c, "reproduction": verdict, "top_A": ta, "top_B": tb,
                         "sign_agreement": round(sa, 3)})
verdict_tbl = pd.DataFrame(verdict_rows)
verdict_tbl.to_csv(TAB / "table_category_reproduction.csv", index=False, encoding="utf-8-sig")
pd.to_pickle({r["category"]: r["reproduction"] for r in verdict_rows}, DATA / "reproduction.pkl")

# ---- (b) LOGO resampling ----
logo_rows = []
for c in ABN:
    if c == "ER":
        continue
    top = top_sensor(full[c])
    if top is None:
        continue
    fs = np.sign(full[c][top])
    pres, sg = [], []
    for leave in used_hh:
        d = agg([h for h in used_hh if h != leave])[c]
        p = top in d
        pres.append(1 if p else 0)
        sg.append(1 if (p and np.sign(d[top]) == fs) else 0)
    logo_rows.append({"category": c, "top_precursor": top,
                      "presence_share": round(float(np.mean(pres)), 3),
                      "sign_stability": round(float(np.mean(sg)), 3)})
logo_tbl = pd.DataFrame(logo_rows)
logo_tbl.to_csv(TAB / "table_logo_stability.csv", index=False, encoding="utf-8-sig")

# ---- (c) Minimum-session-length sensitivity (144 vs 72 steps) ----
HH72 = hh_precursors(72)
full72 = {c: {} for c in ABN}
acc = {c: {} for c in ABN}
for hh, dd in HH72.items():
    for c, d in dd.items():
        for s, v in d.items():
            acc[c].setdefault(s, []).append(v)
full72 = {c: {s: float(np.mean(v)) for s, v in d.items()} for c, d in acc.items()}
param_rows = []
for c in ABN:
    if c == "ER":
        continue
    t1, t2 = top_sensor(full[c]), top_sensor(full72[c])
    param_rows.append({"category": c, "top_144steps": t1 or "(none)", "top_72steps": t2 or "(none)",
                       "same_top": (t1 == t2),
                       "sign_same": (t1 is not None and t2 is not None and t1 in full72[c]
                                     and np.sign(full[c][t1]) == np.sign(full72[c][t1]))})
param_tbl = pd.DataFrame(param_rows)
param_tbl.to_csv(TAB / "table_param_sensitivity.csv", index=False, encoding="utf-8-sig")

# ---- Figure 7. A/B agreement of top-precursor strengths ----
xa, xb, labs = [], [], []
for c in ABN:
    if c == "ER":
        continue
    top = top_sensor(full[c])
    if top and top in pA[c] and top in pB[c]:
        xa.append(abs(pA[c][top])); xb.append(abs(pB[c][top])); labs.append(CAT_LABEL[c])
fig, ax = plt.subplots(figsize=(4.4, 4.2))
if xa:
    ax.scatter(xa, xb, s=42, color=GRAY[1], edgecolor="black", zorder=3)
    lim = max(max(xa), max(xb)) * 1.25
    ax.plot([0, lim], [0, lim], "--", color="0.4", linewidth=1)
    for x, y, l in zip(xa, xb, labs):
        ax.annotate(l, (x, y), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("|partial corr|  split A")
ax.set_ylabel("|partial corr|  split B")
save_fig(fig, "fig07_ab_agreement")

# ---- Figure 8. LOGO sign-stability ----
fig, ax = plt.subplots(figsize=(5.2, 3.2))
if len(logo_tbl):
    ax.bar([CAT_LABEL[c] for c in logo_tbl["category"]], logo_tbl["sign_stability"],
           color=GRAY[2], edgecolor="black", linewidth=0.6)
    ax.axhline(0.9, color="black", linestyle="--", linewidth=1.0, label="0.90")
    ax.legend(frameon=False)
ax.set_ylabel("LOGO sign stability")
ax.set_ylim(0, 1.05)
save_fig(fig, "fig08_logo_sign_stability")

# ---- Console summary ----
print("=== Step 3 (part 2): causal stability of category precursors ===")
print(f"Households: {n_used} | split A={len(A)}, B={len(B)}")
print()
print("(a) Per-category reproduction verdict (A/B internal check):")
print(verdict_tbl.to_string(index=False))
print()
print("(b) LOGO resampling stability:")
print(logo_tbl.to_string(index=False))
print()
print("(c) Minimum-session-length sensitivity (144 vs 72 steps):")
print(param_tbl.to_string(index=False))
print()
print("Figures saved: fig07_ab_agreement, fig08_logo_sign_stability  (png + pdf, 600 dpi)")
print("Tables saved : table_category_reproduction, table_logo_stability, table_param_sensitivity")
print("Artifact     : reproduction.pkl")


=== Step 3 (part 2): causal stability of category precursors ===
Households: 22 | split A=11, B=11

(a) Per-category reproduction verdict (A/B internal check):
category  reproduction           top_A           top_B sign_agreement
      HR          full     Illuminance     Illuminance            0.8
      BR          full     Illuminance     Illuminance            0.9
      ST          full SkinTemperature SkinTemperature            0.8
      SP          full            SPO2            SPO2            0.7
      ER direct_signal    Button/Shout    Button/Shout            n/a

(b) LOGO resampling stability:
category   top_precursor  presence_share  sign_stability
      HR     Illuminance             1.0             1.0
      BR     Illuminance             1.0             1.0
      ST SkinTemperature             1.0             1.0
      SP            SPO2             1.0             1.0

(c) Minimum-session-length sensitivity (144 vs 72 steps):
category    top_144steps     top_72steps  sa